# Implementing augmentation and testing

## Images selection

In [1]:
import numpy as np
import pandas as pd
import random

In [2]:
data = pd.read_csv('data/train.csv')
data.head()

,filename,category
0,112693.jpg,nature
1,133338.jpg,nature
2,130379.jpg,nature
3,100991.jpg,nature
4,075730.jpg,city


In [3]:
data['category'].value_counts()

category
nature    40000
city      40000
faces     40000
Name: count, dtype: int64

In [4]:
def seed_all(seed):
    np.random.seed(seed)
    random.seed(seed)

seed_all(42)

In [5]:
# randomly select n samples from each category
n = 10
sampled_data = data.groupby('category').apply(lambda x: x.sample(n)).reset_index(drop=True)
sampled_data

/var/folders/2s/dztmgkl93_xfxpgb_dc8lwhw0000gn/T/ipykernel_88584/667838076.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_data = data.groupby('category').apply(lambda x: x.sample(n)).reset_index(drop=True)


,filename,category
0,053110.jpg,city
1,062329.jpg,city
2,092885.jpg,city
3,088577.jpg,city
4,098244.jpg,city
5,055755.jpg,city
6,073469.jpg,city
7,080396.jpg,city
8,070633.jpg,city
9,090895.jpg,city


# You must have downloaded the images for next step, if no -> the selected_images directory with the selected images has already been created, just skip the next cell

In [ ]:
# get the files and their categories and copy them to the directory
for i, row in sampled_data.iterrows():
    file_path = row['filename']
    category = row['category']
    image_path = f"data/images/{file_path}"
    selected_path = "data/selected_images"

    with open(image_path, 'rb') as f:
        image_data = f.read()
    with open(f"{selected_path}/{file_path.split('.')[0]}_{category}.jpg", 'wb') as f:
        f.write(image_data)

## Augmentation

### Overview
A comprehensive image augmentation tool for creating degraded images to train ML models for image restoration tasks.

### Features

#### **Degradation Types**
- **Noise**: Gaussian, salt-pepper, speckle
- **Blur**: Gaussian, median, motion blur
- **Resolution**: Downscaling with upscaling artifacts
- **Compression**: JPEG artifacts (quality 20-30)
- **Physical damage**: Scratches and dust spots
- **Color**: Grayscale conversion

#### **Combined Scenarios**
- **`combined_restoration`**: Dust_and_Scratch/None + noise(Gaussian/SP/Speckle) + low resolution/compression
- **`combined_superres`**: Blur(Gaussian/Median/Motion blur) + resolution reduction/compression
- **`combined_colorization`**: Grayscale + Noise(Gaussian/SP/Speckle)/Blur(Gaussian/Median/Motion blur)/None


### Methods

#### Core Methods
| Method | Description | Parameters |
|--------|-------------|------------|
| `add_noise()` | Adds various noise types | `noise_type`, `amount` |
| `resolution_reduce()` | Simulates low resolution | `scale` factor |
| `add_blur()` | Applies different blur types | `blur_type`, `kernel_size` |
| `compress_image()` | JPEG compression artifacts | `quality` level |
| `add_dust_and_scratch()` | Physical damage simulation | Random scratches/dust |

#### Scratch Generation
- **Random**: Short scratches in random locations
- **Edge-to-edge**: Long scratches crossing image boundaries using parametric line equations

In [2]:
import cv2
import numpy as np
import os
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
from skimage import util
import random

In [19]:
class ImageAugmentor:
    def __init__(self, input_dir="data/selected_images", output_dir="data/augmented_images"):
        self.input_dir = input_dir
        self.output_dir = output_dir
        self.create_output_directories()

    def create_output_directories(self):
        subdirs = [
            'low_resolution',
            'blurred', 
            'noisy',
            'grayscale',
            'jpeg_compressed',
            'combined',
            'scratch_and_dust'
        ]
        
        os.makedirs(self.output_dir, exist_ok=True)
        
        for subdir in subdirs:
            os.makedirs(f"{self.output_dir}/{subdir}", exist_ok=True)

    def add_noise(self, image, noise_type='gaussian', a=0.05):
        if noise_type == 'gaussian':
            noise = np.random.normal(0, a * 255, image.shape).astype(np.uint8)
            noisy_image = cv2.add(image, noise)
        elif noise_type == 'salt_pepper':
            noisy_image = util.random_noise(image, mode='s&p', amount=a)
            noisy_image = (255 * noisy_image).astype(np.uint8)
        elif noise_type == 'speckle':
            noise = np.random.randn(*image.shape).astype(np.uint8)
            noisy_image = image + image * noise * a
            noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)
        else:
            raise ValueError("Unsupported noise type")
        return noisy_image

    def resolution_reduce(self, image, scale=0.5):
        height, width = image.shape[:2]
        smaller_image = cv2.resize(image, (int(width * scale), int(height * scale)), interpolation=cv2.INTER_AREA)
        restored_image = cv2.resize(smaller_image, (width, height), interpolation=cv2.INTER_CUBIC)
        return restored_image

    def add_blur(self, image, blur_type='gaussian', ksize=1):
        if blur_type == 'gaussian':
            blurred_image = cv2.GaussianBlur(image, (ksize, ksize), 0)
        elif blur_type == 'median':
            blurred_image = cv2.medianBlur(image, ksize)
        elif blur_type == 'motion':
            kernel = np.zeros((ksize, ksize))
            kernel[int((ksize - 1)/2), :] = np.ones(ksize)
            kernel = kernel / ksize
            blurred_image = cv2.filter2D(image, -1, kernel)
        else:
            raise ValueError("Unsupported blur type")
        return blurred_image

    def compress_image(self, image, quality=20):
        encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
        _, encimg = cv2.imencode('.jpg', image, encode_param)
        compressed = cv2.imdecode(encimg, 1)
        return compressed

    def grayscale(self, image):
        gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        return gray_image

    def add_dust_and_scratch(self, image):
        num_scratches = random.randint(0, 3)
        num_dust = random.randint(10, 20)
        corrupted = image.copy()
        h, w = corrupted.shape[:2]
        
        for _ in range(num_scratches):
            scratch_type = random.choice(['random', 'edge_to_edge'])
            if scratch_type == 'random':
                pt1 = (random.randint(0, w), random.randint(0, h))
                pt2 = (random.randint(0, w), random.randint(0, h))
            else:
                center_x = random.randint(0, w-1)
                center_y = random.randint(0, h-1)
            
                angle = random.uniform(0, 2 * np.pi)
            
                dx = np.cos(angle)
                dy = np.sin(angle)
            
                intersections = []
            
                if dx != 0:
                    t = (w - 1 - center_x) / dx
                    y = center_y + t * dy
                    if 0 <= y <= h - 1:
                        intersections.append((w - 1, int(y)))

                if dx != 0:
                    t = (0 - center_x) / dx
                    y = center_y + t * dy
                    if 0 <= y <= h - 1:
                        intersections.append((0, int(y)))

                if dy != 0:
                    t = (h - 1 - center_y) / dy
                    x = center_x + t * dx
                    if 0 <= x <= w - 1:
                        intersections.append((int(x), h - 1))
                
                if dy != 0:
                    t = (0 - center_y) / dy
                    x = center_x + t * dx
                    if 0 <= x <= w - 1:
                        intersections.append((int(x), 0))

                if len(intersections) >= 2:
                    intersections.sort(key=lambda p: (p[0] - center_x)**2 + (p[1] - center_y)**2)
                    pt1 = intersections[0]
                    pt2 = intersections[-1]
                else:
                    pt1 = (random.randint(0, w-1), random.randint(0, h-1))
                    pt2 = (random.randint(0, w-1), random.randint(0, h-1))
            

            gray_value = random.randint(110, 210)
            color = (gray_value, gray_value, gray_value)
            cv2.line(corrupted, pt1, pt2, color, 1)

        for _ in range(num_dust):
            center = (random.randint(0, w), random.randint(0, h))
            radius = random.randint(1, 2)
            gray_value = random.randint(110, 210)
            color = (gray_value, gray_value, gray_value)
            cv2.circle(corrupted, center, radius, color, -1)
        
        return corrupted

    def process_images(self, augmentation_type=['low_resolution', 'blurred_gaussian', 'blurred_median', 'blurred_motion', 'noisy_gaussian', 'noisy_salt_pepper', 'noisy_speckle', 'grayscale', 'jpeg_compressed', 'dust_and_scratch', 'combined_restauration', 'combined_superres', 'combined_colorization', 'all']):
        for filename in os.listdir(self.input_dir):
            if filename.endswith(".jpg") or filename.endswith(".png"):
                image_path = os.path.join(self.input_dir, filename)
                image = cv2.imread(image_path)
                base_name = os.path.splitext(filename)[0].split('.')[0]
                if 'all' in augmentation_type:
                    augmentation_type = ['low_resolution', 'blurred_gaussian', 'blurred_median', 'blurred_motion', 'noisy_gaussian', 'noisy_salt_pepper', 'noisy_speckle', 'grayscale', 'jpeg_compressed', 'dust_and_scratch', 'combined_restauration', 'combined_superres', 'combined_colorization']

                if 'low_resolution' in augmentation_type:
                    low_res = self.resolution_reduce(image, np.random.choice([0.3, 0.4, 0.5]))
                    cv2.imwrite(f"{self.output_dir}/low_resolution/{base_name}_lowres.jpg", low_res)

                if 'blurred_gaussian' in augmentation_type:
                    blurred_gaussian = self.add_blur(image, 'gaussian', np.random.choice([3,5,7,9]))
                    cv2.imwrite(f"{self.output_dir}/blurred/{base_name}_blur_gaussian.jpg", blurred_gaussian)

                if 'blurred_median' in augmentation_type:
                    blurred_median = self.add_blur(image, 'median', np.random.choice([3,5]))
                    cv2.imwrite(f"{self.output_dir}/blurred/{base_name}_blur_median.jpg", blurred_median)

                if 'blurred_motion' in augmentation_type:
                    blurred_motion = self.add_blur(image, 'motion', np.random.choice([3,5]))
                    cv2.imwrite(f"{self.output_dir}/blurred/{base_name}_blur_motion.jpg", blurred_motion)

                if 'noisy_gaussian' in augmentation_type:
                    noisy_gaussian = self.add_noise(image, 'gaussian', np.random.choice([0.0015, 0.002, 0.001]))
                    cv2.imwrite(f"{self.output_dir}/noisy/{base_name}_noise_gaussian.jpg", noisy_gaussian)

                if 'noisy_salt_pepper' in augmentation_type:
                    noisy_sp = self.add_noise(image, 'salt_pepper', np.random.choice([0.005, 0.01, 0.015, 0.02]))
                    cv2.imwrite(f"{self.output_dir}/noisy/{base_name}_noise_sp.jpg", noisy_sp)
                
                if 'noisy_speckle' in augmentation_type:
                    noisy_speckle = self.add_noise(image, 'speckle', np.random.choice([0.05, 0.1, 0.15, 0.2, 0.25, 0.3]))
                    cv2.imwrite(f"{self.output_dir}/noisy/{base_name}_noise_speckle.jpg", noisy_speckle)

                if 'grayscale' in augmentation_type:
                    grayscale = self.grayscale(image)
                    cv2.imwrite(f"{self.output_dir}/grayscale/{base_name}_grayscale.jpg", grayscale)

                if 'jpeg_compressed' in augmentation_type:
                    compressed = self.compress_image(image, np.random.choice([20, 30]))
                    cv2.imwrite(f"{self.output_dir}/jpeg_compressed/{base_name}_compressed.jpg", compressed)

                if 'dust_and_scratch' in augmentation_type:
                    dust_scratch = self.add_dust_and_scratch(image)
                    cv2.imwrite(f"{self.output_dir}/scratch_and_dust/{base_name}_dust_scratch.jpg", dust_scratch)
                
                if 'combined_restauration' in augmentation_type:
                    if random.random() < 0.4:
                        combined_restoration = self.add_dust_and_scratch(image)
                    else:
                        combined_restoration = image.copy()

                    n = random.random()
                    if n < 0.5:
                        combined_restoration = self.add_noise(combined_restoration, 'speckle', np.random.choice([0.05, 0.1, 0.15, 0.2, 0.25, 0.3]))
                    elif n < 0.75:
                        combined_restoration = self.add_noise(combined_restoration, 'salt_pepper', np.random.choice([0.005, 0.01, 0.015, 0.02]))
                    else:
                        combined_restoration = self.add_noise(combined_restoration, 'gaussian', np.random.choice([0.0015, 0.002, 0.001]))
                    
                    n = random.random()
                    if n < 0.33:
                        combined_restoration = self.resolution_reduce(combined_restoration, np.random.choice([0.3, 0.4, 0.5]))
                    elif n < 0.66:
                        combined_restoration = self.compress_image(combined_restoration, np.random.choice([20, 30]))
                    
                    cv2.imwrite(f"{self.output_dir}/combined/{base_name}_for_restoration.jpg", combined_restoration)
                
                if 'combined_superres' in augmentation_type:
                    n = random.random()
                    if n < 0.5:
                        super_low_res = self.add_blur(image, 'motion', np.random.choice([3,5]))
                    elif n < 0.75:
                        super_low_res = self.add_blur(image, 'median', np.random.choice([3,5]))
                    else:
                        super_low_res = blurred_gaussian = self.add_blur(image, 'gaussian', np.random.choice([3,5,7,9]))
                    n = random.random()
                    if n < 0.5:
                        super_low_res = self.resolution_reduce(super_low_res, np.random.choice([0.3, 0.4, 0.5]))
                    else:
                        super_low_res = self.compress_image(super_low_res, np.random.choice([20, 30]))

                    cv2.imwrite(f"{self.output_dir}/combined/{base_name}_for_superres.jpg", super_low_res)

                if 'combined_colorization' in augmentation_type:
                    for_colorization = self.grayscale(image)
                    n = random.random()
                    if n < 0.4:
                        n = random.random()
                        if n < 0.5:
                            for_colorization = self.add_noise(for_colorization, 'speckle', np.random.choice([0.05, 0.1]))
                        elif n < 0.75:
                            for_colorization = self.add_noise(for_colorization, 'salt_pepper', np.random.choice([0.005, 0.01]))
                        else:
                            for_colorization = self.add_noise(for_colorization, 'gaussian', np.random.choice([0.0015, 0.001]))
                    elif n < 0.8:
                        n = random.random()
                        if n < 0.5:
                            for_colorization = self.add_blur(for_colorization, 'motion', 3)
                        elif n < 0.75:
                            for_colorization = self.add_blur(for_colorization, 'median', 3)
                        else:
                            for_colorization = blurred_gaussian = self.add_blur(for_colorization, 'gaussian', np.random.choice([3,5]))

                    cv2.imwrite(f"{self.output_dir}/combined/{base_name}_for_colorization.jpg", for_colorization)
                
                
                print(f"Image processed: {filename}")


In [20]:
corruptor = ImageAugmentor()
corruptor.process_images(augmentation_type=['all'])

Image processed: 020513_faces.jpg
Image processed: 042885_faces.jpg
Image processed: 083370_city.jpg
Image processed: 029735_faces.jpg
Image processed: 085715_city.jpg
Image processed: 113738_nature.jpg
Image processed: 046477_faces.jpg
Image processed: 095820_city.jpg
Image processed: 122530_nature.jpg
Image processed: 041558_faces.jpg
Image processed: 130674_nature.jpg
Image processed: 106968_nature.jpg
Image processed: 080418_city.jpg
Image processed: 094144_city.jpg
Image processed: 111253_nature.jpg
Image processed: 067788_city.jpg
Image processed: 126135_nature.jpg
Image processed: 018197_faces.jpg
Image processed: 105356_nature.jpg
Image processed: 063400_city.jpg
Image processed: 090743_city.jpg
Image processed: 103830_nature.jpg
Image processed: 077612_city.jpg
Image processed: 036033_faces.jpg
Image processed: 008749_faces.jpg
Image processed: 069279_city.jpg
Image processed: 019092_faces.jpg
Image processed: 049132_faces.jpg
Image processed: 144838_nature.jpg
Image processed